# Notebook 1 · revisão 04 — módulo IFC2SB parcial em Python

Complemento executável à conversão IFC–LBD da revisão 03. **Objetivo:** testar rotinas geométricas publicadas em um piloto da cozinha, preservar coordenadas no RDF e distinguir candidatos, separações e interpenetrações.

Fonte científica: [Fichter et al. (2021)](https://doi.org/10.26868/25222708.2021.30156). [Código dos autores, commit fixado](https://github.com/RWTH-E3D/IFC2SB/tree/96a89cdff65f13279c0c3ce67e729ad9f9f5adad). [Mapa das funções, diferenças e licença](https://github.com/prcgui/bim-ontology-CIV2155/tree/main/consultas/ifc2sb_python_v01).

Trata-se de portabilidade parcial com uma extensão ambiente-face explicitada, não de reconstrução completa ou arquitetura validada. O IFC original permanece intacto.

## 1. Preparar as fontes

O módulo pode ser executado nesta pasta ou no Colab. Arquivos baixados são confrontados com hashes fixados nesta versão do notebook. Para um ambiente novo, são instaladas as versões indicadas; ambientes com versões diferentes devem ser reiniciados e preparados conforme `requirements.txt`.

In [1]:
# Preparação técnica: bibliotecas fixadas e fontes verificadas por SHA-256.
# Referências científicas e diferenças: README.md do módulo (célula anterior).
from pathlib import Path
import sys,subprocess,importlib.util,urllib.request,hashlib,json
local=Path.cwd()
if (local/'ifc2sb_faces.py').exists():
    modulo=local
else:
    modulo=local/'ifc2sb_python_v01';modulo.mkdir(exist_ok=True)
EXPECTED={'ifc2sb_faces.py': '078cbc2c93c6f81854eb26738ddf26d31d34d3196c7533b8e54e734113d6a155', 'piloto.py': '71cd473ccc9bfdda5f8d89982046ca37e5991341e5412d7e4cb1c6e3b6fb828a', 'exportar_rdf.py': 'dc2e81194d029445f2697c2f6ab2b04458355915021176a49dbf7af218ebfda1', 'testar_adaptacao.py': 'ad4a1024b82130cfc5399b1dc3dd7ebf4b12cda1b1d82f53233db3097f7c984a', 'referencia_cpp/Resultado_CPP.json': '2e7cfc73670a63e77db9ce4cf0560dea2c51e5d2cc93ad1762be68bd08f83332', 'README.md': '5b70d51f757b14a28c0e9f96152d684b10c3056f6d56ad3ed73f724045ac7171', 'LICENSE': '3972dc9744f6499f0f9b2dbf76696f2ae7ad8af9b23dde66d6af86c9dfb36986', 'Manifesto_Fontes.json': 'dd994dce951b0d70b2deab489f80b8042fce0864aa3ffc55bc9e93cc29a7c292'}
url='https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/main/consultas/ifc2sb_python_v01/'
for name,digest in EXPECTED.items():
    target=modulo/name;target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists():urllib.request.urlretrieve(url+name,target)
    assert hashlib.sha256(target.read_bytes()).hexdigest()==digest, f'Fonte divergente: {name}'
for package,distribution in [('ifcopenshell','ifcopenshell==0.8.4'),('OCP','cadquery-ocp==7.8.1.1.post1'),('rdflib','rdflib==7.1.4')]:
    if importlib.util.find_spec(package) is None:subprocess.check_call([sys.executable,'-m','pip','install',distribution])
sys.path.insert(0,str(modulo.resolve()))
import ifcopenshell,OCP,rdflib
assert ifcopenshell.version=='0.8.4', 'Usar o ambiente de versões registrado'
assert OCP.__version__.startswith('7.8.1'), 'Usar OCP 7.8.1.1.post1'
assert rdflib.__version__=='7.1.4', 'Usar RDFLib 7.1.4'
saida=modulo/'saida_executada';saida.mkdir(exist_ok=True)
print('Fontes verificadas:',len(EXPECTED),'| IfcOpenShell',ifcopenshell.version,'| OCP',OCP.__version__,'| RDFLib',rdflib.__version__)


Fontes verificadas: 8 | IfcOpenShell 0.8.4 | OCP 7.8.1.1 | RDFLib 7.1.4


## 2. Conferir a adaptação com controles

Os corpos C++ originais foram compilados e executados em ensaio isolado. Aqui se repetem os 17 controles Python e se confrontam nove resultados com essa referência registrada. Não se recompila o programa C++ nesta célula. [Funções originais e hashes](https://github.com/prcgui/bim-ontology-CIV2155/blob/main/consultas/ifc2sb_python_v01/Extracao_Corpos_CPP.json).

Um controle demonstra que `CheckParent` pode aceitar uma face coplanar fora do contorno do hospedeiro. Por isso, esse teste não substitui a interseção com o ambiente.

In [2]:
from testar_adaptacao import run as testar
controles=testar(modulo/'referencia_cpp/Resultado_CPP.json',saida/'controles')
assert controles['aprovados']==17


CONTROLES 17 / 17


## 3. Importar o IFC original

Fonte: projeto arquitetônico R03 da Creche Tipo 2, FNDE, preservado no repositório. A verificação SHA-256 impede a substituição silenciosa do arquivo. O caminho local é utilizado quando disponível; no Colab, a mesma entrada é baixada do repositório.

[Fonte institucional](https://www.gov.br/fnde/pt-br/acesso-a-informacao/acoes-e-programas/programas/proinfancia/projetos-arquitetonicos-para-construcao/projeto-tipo-2) · [Processamento geométrico IfcOpenShell](https://docs.ifcopenshell.org/ifcopenshell-python/geometry_processing.html).

In [3]:
from piloto import IFC_SHA
caminho_local=Path('C:/Users/prcgu/OneDrive/Área de Trabalho/artigo bim CIV2155/02_Dados_BIM/Arquitetura/TIPO2-ARQ-DTV_R03.ifc')
ifc=caminho_local if caminho_local.exists() else modulo/'TIPO2-ARQ-DTV_R03.ifc'
if not ifc.exists():urllib.request.urlretrieve('https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/main/dados/fnde/arquitetura/TIPO2-ARQ-DTV_R03.ifc',ifc)
assert hashlib.sha256(ifc.read_bytes()).hexdigest()==IFC_SHA
print(ifc.name,'| SHA-256 confirmado |',ifc.stat().st_size,'bytes')


TIPO2-ARQ-DTV_R03.ifc | SHA-256 confirmado | 27852645 bytes


## 4. Executar o piloto da cozinha

Rotinas de origem: `cvfcipw_*`, `CheckParent`, `face_normal`, `unify_shape` e o subconjunto Sewing/MakeSolid do IFC2SB. Extensão do estudo: `room_face_common`, com interseção OCCT em planos paralelos, em lugar de Clipper; a equivalência geral não é afirmada.

Todas as paredes são examinadas no pré-filtro. Bbox seleciona candidatas; a relação é investigada por face. O espaço precisa formar sólido fechado e válido antes de se interpretar volume. Casos de interpenetração ficam registrados sem solução. [Detalhamento da adaptação](https://github.com/prcgui/bim-ontology-CIV2155/blob/main/consultas/ifc2sb_python_v01/README.md).

In [4]:
from piloto import run as executar_piloto
from collections import Counter
resultado=executar_piloto(ifc,saida/'piloto')
print('Topologia:',resultado['topologia_ambiente'])
print('Estados das aberturas:',dict(Counter(r['estado'] for r in resultado['aberturas'])))
assert resultado['original_preservado'] and not resultado['falhas']


W01 0XAHc8zyr1YuL0AWxiP2D2 aberturas 5 faces 104 contato_bruto 10.012775731856873


W02 3lgLMxPtX22hHa1J_NWBcS aberturas 1 faces 18 contato_bruto 0.0


W03 2gY30MgeHFYR7UxJlNnU9q aberturas 0 faces 0 contato_bruto 18.600002762841427


W04 0xvlN6VObElAHCGt3ouxRe aberturas 1 faces 2 contato_bruto 10.85209814920372


W05 1v8GUEVp5BSuN5s4rGKaRA aberturas 1 faces 2 contato_bruto 6.052861845429635


...

W32 0m8CetgqbC3OnOpkIkk9rr aberturas 0 faces 0 contato_bruto 0.0


W33 0sj_aEsMzBlPSsBHpOD6vZ aberturas 0 faces 0 contato_bruto 0.0


W34 1TlNCt8aP6oesbR1Y5mQco aberturas 0 faces 0 contato_bruto 0.5269999991913245


RESUMO Counter({'sem_intersecao_de_face_no_criterio': 11, 'evidencia_geometrica_de_coincidencia': 6, 'coincidencia_com_interpenetracao_parede_ambiente': 4, 'fora_do_ambiente_por_separacao_bbox': 3}) FALHAS []


Topologia: {'faces_entrada': 23, 'faces_saida': 23, 'arestas_livres': 0, 'solidos_entrada': 0, 'solidos_saida': 1, 'volume_m3': 109.52494948221211, 'procedimento': 'subconjunto Sewing/MakeSolid do IFC2SB; sem remoção de faces nem reparação global'}
Estados das aberturas: {'fora_do_ambiente_por_separacao_bbox': 3, 'evidencia_geometrica_de_coincidencia': 6, 'sem_intersecao_de_face_no_criterio': 11, 'coincidencia_com_interpenetracao_parede_ambiente': 4}


## 5. Gravar e consultar as relações com coordenadas no RDF

A serialização geométrica considera a proposta de [BimSPARQL, Zhang et al. (2018)](https://doi.org/10.3233/SW-180297). A exportação e a consulta específicas são deste estudo, utilizando [RDFLib](https://rdflib.readthedocs.io/en/stable/intro_to_graphs/).

Cadeia consultável: **ambiente → superfície bruta candidata → recorte de abertura → abertura → preenchimento**. O grafo guarda WKT XYZ, metros, referencial do projeto, GUIDs e estado da evidência. Candidatos em conflito não são convertidos em afirmações arquitetônicas validadas. Não se usam coordenadas geográficas implícitas.

A consulta é executada e comparada com os registros geométricos. Os casos separados por bbox devem permanecer fora das cadeias de recortes coincidentes.

In [5]:
from exportar_rdf import exportar
registro=exportar(saida/'piloto/Resultado_Piloto.json',saida/'piloto')
print(json.dumps(registro,ensure_ascii=False,indent=2))
print((saida/'piloto/Consulta_Recortes.csv').read_text(encoding='utf-8-sig')[:3500])


{'triplas': 676, 'superficies_candidatas': 14, 'recortes_coincidentes': 11, 'aberturas_com_recortes': 10, 'aberturas_separadas_sem_cadeia': 3, 'consulta_linhas': 11, 'sha256_RDF': '402dd74f390f60ea583089a4af5f50341a667d8aa3da21cb64cb623d5d519677', 'limite': 'Grafo de evidências candidatas; não é atribuição arquitetônica validada. Relações declaradas IFC são distintas das superfícies calculadas.'}
{
  "triplas": 676,
  "superficies_candidatas": 14,
  "recortes_coincidentes": 11,
  "aberturas_com_recortes": 10,
  "aberturas_separadas_sem_cadeia": 3,
  "consulta_linhas": 11,
  "sha256_RDF": "402dd74f390f60ea583089a4af5f50341a667d8aa3da21cb64cb623d5d519677",
  "limite": "Grafo de evidências candidatas; não é atribuição arquitetônica validada. Relações declaradas IFC são distintas das superfícies calculadas."
}
ambiente,superficie,parede,abertura,preenchimento,estado,coordenadas
https://github.com/prcgui/bim-ontology-CIV2155/id/ifc2sb-python-v01/3UFjAT4k55zg_Pqo6VMw5l,https://github.com/prc

## 6. Limites e integridade

O piloto não demonstra completude dos 41 ambientes nem validação arquitetônica. Não abrange fallback por caixas, faces curvas ou reconstrução global. Superfície bruta não equivale a área líquida da parede. A presença de coordenadas auxilia a integração, mas não resolve diferenças de referenciais, conflitos ou ausência de informação.

As tolerâncias e as diferenças em relação ao código publicado estão nas fontes. A interpretação arquitetônica dos candidatos e dos conflitos permanece pendente. O próximo teste de expansão deve preservar os mesmos controles e registrar os ambientes inconclusivos.

In [6]:
assert hashlib.sha256(ifc.read_bytes()).hexdigest()==IFC_SHA
resumo={'controles':controles['aprovados'],'paredes_examinadas':resultado['paredes_examinadas'],
 'paredes_candidatas':resultado['paredes_candidatas_bbox'],'aberturas_examinadas':len(resultado['aberturas']),
 'estados':dict(Counter(r['estado'] for r in resultado['aberturas'])),'rdf':registro,
 'original_preservado':True,'validacao_arquitetonica':'pendente'}
(saida/'Resumo_Execucao.json').write_text(json.dumps(resumo,ensure_ascii=False,indent=2),encoding='utf8')
print(json.dumps(resumo,ensure_ascii=False,indent=2))


{
  "controles": 17,
  "paredes_examinadas": 457,
  "paredes_candidatas": 34,
  "aberturas_examinadas": 24,
  "estados": {
    "fora_do_ambiente_por_separacao_bbox": 3,
    "evidencia_geometrica_de_coincidencia": 6,
    "sem_intersecao_de_face_no_criterio": 11,
    "coincidencia_com_interpenetracao_parede_ambiente": 4
  },
  "rdf": {
    "triplas": 676,
    "superficies_candidatas": 14,
    "recortes_coincidentes": 11,
    "aberturas_com_recortes": 10,
    "aberturas_separadas_sem_cadeia": 3,
    "consulta_linhas": 11,
    "sha256_RDF": "402dd74f390f60ea583089a4af5f50341a667d8aa3da21cb64cb623d5d519677",
    "limite": "Grafo de evidências candidatas; não é atribuição arquitetônica validada. Relações declaradas IFC são distintas das superfícies calculadas."
  },
  "original_preservado": true,
  "validacao_arquitetonica": "pendente"
}
